In [0]:

# Bronze -> Silver for `commitment` (internal CSV, 66 rows).
# First fact table with foreign keys - run AFTER 01_silver_fund and


In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from datetime import date

SOURCE_NAME = "commitment"
REQUIRED_COLS = ["commitment_id", "fund_id", "investor_id", "committed_amount_usd", "commitment_date"]
KEY_COLS = ["commitment_id"]
COMPARE_COLS = ["fund_id", "investor_id", "committed_amount_usd", "commitment_date"]
business_date_str = date.today().isoformat()

In [0]:
bronze_df = read_bronze(spark, SOURCE_NAME)
print(f"Bronze row count: {bronze_df.count()}")

Bronze row count: 330


In [0]:
typed_df = (
    bronze_df
    .withColumn("commitment_id", F.trim(F.col("commitment_id")))
    .withColumn("fund_id", F.trim(F.col("fund_id")))
    .withColumn("investor_id", F.trim(F.col("investor_id")))
    .withColumn("committed_amount_usd", F.col("committed_amount_usd").cast("double"))
    .withColumn("commitment_date", F.to_date("commitment_date"))
    .withColumn("contributed_total", F.col("contributed_total").cast("double"))
    .withColumn("invested_total", F.col("invested_total").cast("double"))
)

In [0]:
clean_df, null_rejects_df = split_on_required_nulls(typed_df, REQUIRED_COLS)
null_reject_count = null_rejects_df.count()
if null_reject_count > 0:
    write_quarantine(null_rejects_df, SOURCE_NAME)

### Referential integrity - fund_id and investor_id must resolve

In [0]:
fund_silver = spark.table(silver_table("fund"))
investor_silver = spark.table(silver_table("investor"))

valid_fk_df, invalid_fund_fk_df = check_foreign_key(clean_df, "fund_id", fund_silver, "fund_id")
valid_fk_df, invalid_investor_fk_df = check_foreign_key(valid_fk_df, "investor_id", investor_silver, "investor_id")

invalid_fk_count = invalid_fund_fk_df.count() + invalid_investor_fk_df.count()
if invalid_fund_fk_df.count() > 0:
    write_quarantine(invalid_fund_fk_df, SOURCE_NAME)
if invalid_investor_fk_df.count() > 0:
    write_quarantine(invalid_investor_fk_df, SOURCE_NAME)

In [0]:
deduped_df, duplicates_df, breaks_df = split_duplicates(valid_fk_df, KEY_COLS, COMPARE_COLS)
dup_count = duplicates_df.count()
break_count = breaks_df.count()
if dup_count > 0:
    write_quarantine(duplicates_df, SOURCE_NAME)
if break_count > 0:
    write_quarantine(breaks_df.withColumn("reason_code", F.lit("COMMITMENT_ATTRIBUTE_BREAK")), SOURCE_NAME)

In [0]:
write_silver(deduped_df, SOURCE_NAME)
print(f"Silver row count: {deduped_df.count()}")

Silver row count: 66


In [0]:
log_dq(spark, SOURCE_NAME, business_date_str, "null_required_field", bronze_df.count(), null_reject_count, "NULL_REQUIRED_FIELD")
log_dq(spark, SOURCE_NAME, business_date_str, "unknown_reference", clean_df.count(), invalid_fk_count, "UNKNOWN_REFERENCE")
log_dq(spark, SOURCE_NAME, business_date_str, "duplicate_record", valid_fk_df.count(), dup_count, "DUPLICATE_RECORD")
log_dq(spark, SOURCE_NAME, business_date_str, "attribute_break", valid_fk_df.count(), break_count, "COMMITMENT_ATTRIBUTE_BREAK")

/home/spark-20788a93-e1e3-423a-8139-90/.ipykernel/71/command-5696143635338715-3986853518:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


In [0]:
bronze_count = bronze_df.count()
silver_count = deduped_df.count()
quarantined_count = null_reject_count + invalid_fk_count + dup_count
assert bronze_count == silver_count + quarantined_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={silver_count}, quarantined={quarantined_count}"
)
print(f"OK: bronze={bronze_count} = silver={silver_count} + quarantined={quarantined_count}")

OK: bronze=330 = silver=66 + quarantined=264
